# Weather Stations to FSA

In [86]:
import pandas as pd
import numpy as np
#os is used for MacOS paths
import os
#The glob module is used to find all file names matching a pattern,
from glob import glob

import geopandas as gpd
from shapely.geometry import Point

# Load data, filter to Ontario

In [103]:
folder_path = '/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/Weather stations to FSAs'

# --- Step 1: Load the FSA shapefile ---
path_to_shapefile = (os.path.join(folder_path, 'Converted CRS Shp file', 'fsa_4326.shp'))
fsa_gdf = gpd.read_file(path_to_shapefile)
fsa_gdf = fsa_gdf[fsa_gdf['PRNAME']=='Ontario']

# --- Step 2: Load the weather stations CSV ---
path_to_weather_station = os.path.join(folder_path, 'Station Inventory EN.csv')
weather_df = pd.read_csv(path_to_weather_station, skiprows=3)
#Just ontario
weather_df = weather_df[weather_df['Province']=='ONTARIO']
# Just get Hourly stations
# Ensure the column is numeric (may be string if imported from CSV)
weather_df['HLY First Year'] = pd.to_numeric(weather_df['HLY First Year'])
# Filter: only keep rows with HLY First Year >= 2023 and no NaN
weather_df = weather_df[weather_df['HLY First Year'].notna() & (weather_df['HLY First Year'] <= 2023)]

# Convert to GeoDataFrame
weather_gdf = gpd.GeoDataFrame(
    weather_df,
    geometry=gpd.points_from_xy(weather_df['Longitude (Decimal Degrees)'], weather_df['Latitude (Decimal Degrees)']),
    crs="EPSG:4326"  # I converted the original shp file from EPSG:3347 to EPSG:4326. Use the CRS that matches your FSA shapefile
)


In [88]:
weather_df.head()

,Name,Province,Climate ID,Station ID,WMO ID,TC ID,Latitude (Decimal Degrees),Longitude (Decimal Degrees),Latitude,Longitude,Elevation (m),First Year,Last Year,HLY First Year,HLY Last Year,DLY First Year,DLY Last Year,MLY First Year,MLY Last Year
5131,ATTAWAPISKAT A,ONTARIO,6010400,52918,71653.0,YAT,52.93,-82.43,525539000,-822555000,9.1,2015,2025,2015.0,2025.0,2015.0,2025.0,NaN,NaN
5132,BIG TROUT LAKE A,ONTARIO,6010735,49488,71675.0,YTL,53.82,-89.90,534904000,-895349000,222.5,2011,2025,2011.0,2025.0,2011.0,2025.0,NaN,NaN
5133,BIG TROUT LAKE,ONTARIO,6010738,3914,NaN,NaN,53.83,-89.87,535000000,-895200000,224.1,1939,1992,1953.0,1992.0,1939.0,1992.0,1939.0,1992.0
5134,BIG TROUT LAKE READAC,ONTARIO,6010739,10804,NaN,TBL,53.82,-89.90,534856000,-895344000,222.2,1992,2013,1994.0,2013.0,1992.0,2013.0,NaN,NaN
5135,BIG TROUT LAKE,ONTARIO,6010740,50857,71208.0,OTL,53.82,-89.89,534858000,-895330000,222.5,2012,2025,2012.0,2025.0,2012.0,2025.0,NaN,NaN


# Dataframe with all Weather stations mapped to Boundary

In [89]:

# --- Step 3: Ensure both GeoDataFrames use the same CRS ---
if fsa_gdf.crs != weather_gdf.crs:
    weather_gdf = weather_gdf.to_crs(fsa_gdf.crs)

# --- Step 4: Spatial join to find which stations fall in which FSA ---
joined = gpd.sjoin(weather_gdf, fsa_gdf, how="inner", predicate="within")

# --- Step 5: Group by FSA and pivot into multiple columns ---
# Replace 'CFSAUID' and 'Station ID' with actual column names
grouped = (
    joined.groupby('CFSAUID')['Climate ID']
    .apply(list)
    .reset_index()
)

# --- Step 6: Expand station lists into separate columns ---
if grouped.empty or grouped['Climate ID'].dropna().empty:
    print("No matching stations found in any FSA.")
    max_len = 0
    stations_df = pd.DataFrame()
else:
    max_len = int(grouped['Climate ID'].apply(lambda x: len(x) if isinstance(x, list) else 0).max())
    stations_df = pd.DataFrame(
        grouped['Climate ID'].apply(lambda x: x if isinstance(x, list) else []).to_list(),
        columns=[f"weather_station_{i+1}" for i in range(max_len)]
    )

# --- Step 7: Save to CSV ---
result_df = pd.concat([grouped['CFSAUID'], stations_df], axis=1)

# Export to CSV
result_df.to_csv("/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/output_data/fsa_weather_station_mapping.csv", index=False)

In [90]:
len(weather_df)

313

In [91]:
result_df.head()

,CFSAUID,weather_station_1,weather_station_2,weather_station_3,weather_station_4,weather_station_5,weather_station_6,weather_station_7,weather_station_8,weather_station_9,...,weather_station_26,weather_station_27,weather_station_28,weather_station_29,weather_station_30,weather_station_31,weather_station_32,weather_station_33,weather_station_34,weather_station_35
0,K0B,6100176,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,K0C,6100166,6105397,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,K0E,6103024,6103808,6109517,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,K0G,6104027,6106022,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,K0J,6104125,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


# Data with one Weather station per FSA boundary

In [104]:
## This script maps FSA boundaries to weather stations
# If there are multiple weather stations within a boundary it uses the most central weather station within the boundary
# if there are no weather stations in the FSA boundary it uses the closest weather station to the centre of the boundary

# Step 1: Project FSAs and compute centroids
fsa_proj = fsa_gdf.to_crs("EPSG:3347")
fsa_proj['centroid'] = fsa_proj.geometry.centroid

# Step 2: Project weather stations
weather_proj = weather_gdf.to_crs("EPSG:3347")

# Step 3: Spatial join - get all weather stations within each FSA
fsa_with_stations = gpd.sjoin(fsa_proj, weather_proj, how='left', predicate='contains')

# Step 4: Build mapping for each FSA
def get_station_assignment(group, weather_stations):
    num_matches = group['Climate ID'].notna().sum()
    
    if num_matches == 1:
        # Use the one and only matched station
        return pd.Series({
            #select the first row and return CFSAUID
            'CFSAUID': group['CFSAUID'].iloc[0],
            #select the first row and return Climate ID
            'central_weather_station': group['Climate ID'].dropna().iloc[0]
        })
    else:
        # Fallback to nearest to centroid if num_matches == 1 (so 0 or >1 weather stations
        #select first row centroid
        centroid = group['centroid'].iloc[0]
        #calculate distances of weather stations to centroid
        distances = weather_stations.geometry.distance(centroid)
        #get index of the nearest weather station
        nearest_idx = distances.idxmin()
        # get Climate ID of weather station that is nearest 
        nearest_station = weather_stations.loc[nearest_idx, 'Climate ID']
        return pd.Series({
            'CFSAUID': group['CFSAUID'].iloc[0],
            'central_weather_station': nearest_station
        })

# Step 5: Apply function to each FSA group
# use lambda to apply function to each entry of the GroupbyDataframe object 
station_mapping = fsa_with_stations.groupby('CFSAUID').apply(
    lambda group: get_station_assignment(group, weather_proj)
).reset_index(drop=True)

station_mapping = station_mapping.rename(columns={
    "CFSAUID": "FSA",
    "central_weather_station": "Climate ID"
})

# Step 7: add name and station ID to dataframe to be referenced when pulling weather data
station_mapping = station_mapping.merge(
    weather_df[['Climate ID', 'Name', 'Station ID']],
    on='Climate ID',
    how='left'
)

# Step 6: Export result
station_mapping.to_csv(
    "/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/output_data/fsa_central_weather_station_mapping.csv",
    index=False
)


/var/folders/06/r1b4q53n1z9c4xqz_6bqq9zc0000gn/T/ipykernel_29217/3951374923.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  station_mapping = fsa_with_stations.groupby('CFSAUID').apply(


# Secondary Mapping post analysing missing weather stations

In [105]:
folder_path = '/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/Weather stations to FSAs'

# --- Step 1: Load the FSA shapefile ---
path_to_shapefile = (os.path.join(folder_path, 'Converted CRS Shp file', 'fsa_4326.shp'))
fsa_gdf = gpd.read_file(path_to_shapefile)
fsa_gdf = fsa_gdf[fsa_gdf['PRNAME']=='Ontario']

# --- Step 2: Load the weather stations CSV ---
path_to_weather_station = os.path.join(folder_path, 'Remaining_weather_stations.csv')
weather_df = pd.read_csv(path_to_weather_station)

# Convert to GeoDataFrame
weather_gdf = gpd.GeoDataFrame(
    weather_df,
    geometry=gpd.points_from_xy(weather_df['Longitude (Decimal Degrees)'], weather_df['Latitude (Decimal Degrees)']),
    crs="EPSG:4326"  # I converted the original shp file from EPSG:3347 to EPSG:4326. Use the CRS that matches your FSA shapefile
)


In [106]:
## This script maps FSA boundaries to weather stations
# If there are multiple weather stations within a boundary it uses the most central weather station within the boundary
# if there are no weather stations in the FSA boundary it uses the closest weather station to the centre of the boundary

# Step 1: Project FSAs and compute centroids
fsa_proj = fsa_gdf.to_crs("EPSG:3347")
fsa_proj['centroid'] = fsa_proj.geometry.centroid

# Step 2: Project weather stations
weather_proj = weather_gdf.to_crs("EPSG:3347")

# Step 3: Spatial join - get all weather stations within each FSA
fsa_with_stations = gpd.sjoin(fsa_proj, weather_proj, how='left', predicate='contains')

# Step 4: Build mapping for each FSA
def get_station_assignment(group, weather_stations):
    num_matches = group['Climate ID'].notna().sum()
    
    if num_matches == 1:
        # Use the one and only matched station
        return pd.Series({
            #select the first row and return CFSAUID
            'CFSAUID': group['CFSAUID'].iloc[0],
            #select the first row and return Climate ID
            'central_weather_station': group['Climate ID'].dropna().iloc[0]
        })
    else:
        # Fallback to nearest to centroid if num_matches == 1 (so 0 or >1 weather stations
        #select first row centroid
        centroid = group['centroid'].iloc[0]
        #calculate distances of weather stations to centroid
        distances = weather_stations.geometry.distance(centroid)
        #get index of the nearest weather station
        nearest_idx = distances.idxmin()
        # get Climate ID of weather station that is nearest 
        nearest_station = weather_stations.loc[nearest_idx, 'Climate ID']
        return pd.Series({
            'CFSAUID': group['CFSAUID'].iloc[0],
            'central_weather_station': nearest_station
        })

# Step 5: Apply function to each FSA group
# use lambda to apply function to each entry of the GroupbyDataframe object 
station_mapping = fsa_with_stations.groupby('CFSAUID').apply(
    lambda group: get_station_assignment(group, weather_proj)
).reset_index(drop=True)

station_mapping = station_mapping.rename(columns={
    "CFSAUID": "FSA",
    "central_weather_station": "Climate ID"
})

# Step 7: add name and station ID to dataframe to be referenced when pulling weather data
station_mapping = station_mapping.merge(
    weather_df[['Climate ID', 'Name', 'Station ID']],
    on='Climate ID',
    how='left'
)

# Step 6: Export result
station_mapping.to_csv(
    "/Users/will/Documents/Beacon MVP/Projects/IESO Data Set Up/output_data/remaining_fsa_central_weather_station_mapping.csv",
    index=False
)


/var/folders/06/r1b4q53n1z9c4xqz_6bqq9zc0000gn/T/ipykernel_29217/2732788545.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  station_mapping = fsa_with_stations.groupby('CFSAUID').apply(


520